In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os, glob, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T

def _find_first_dir(root, candidates):
    root = Path(root)
    for c in candidates:
        p = root / c
        if p.exists() and p.is_dir():
            return str(p)
    for d in root.rglob("*"):
        if d.is_dir():
            name = d.name.lower()
            if any(k in name for k in candidates):
                return str(d)
    return None

def _list_images(folder):
    exts = ("*.jpg","*.jpeg","*.png","*.bmp")
    files=[]
    for e in exts:
        files.extend(glob.glob(os.path.join(folder,e)))
    return sorted(files)

def build_global_mask_mapping(mask_files):
    vals=set()
    for mf in mask_files:
        m=np.array(Image.open(mf))
        vals.update(np.unique(m).tolist())
    keys=sorted(list(vals))
    mapping={old:new for new,old in enumerate(keys)}
    return mapping, keys

def apply_global_mapping(mask_tensor,mapping):
    out=torch.zeros_like(mask_tensor)
    for old,new in mapping.items():
        out[mask_tensor==old]=new
    return out

class SUIMDataset(Dataset):
    def __init__(self,root_dir,img_size=256):
        self.images_dir=_find_first_dir(root_dir,["images","image","imgs","img"])
        self.masks_dir=_find_first_dir(root_dir,["masks","mask","labels","label","annotations"])
        self.image_files=_list_images(self.images_dir)
        self.mask_files=_list_images(self.masks_dir)
        imap={Path(f).stem:f for f in self.image_files}
        mmap={Path(f).stem:f for f in self.mask_files}
        common=sorted(list(set(imap.keys()) & set(mmap.keys())))
        self.pairs=[(imap[s],mmap[s]) for s in common]
        self.mapping,_=build_global_mask_mapping([m for _,m in self.pairs])
        self.img_tf=T.Compose([T.Resize((img_size,img_size)),T.ToTensor()])
        self.mask_resize=T.Resize((img_size,img_size),interpolation=T.InterpolationMode.NEAREST)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self,idx):
        img_path,mask_path=self.pairs[idx]
        img=self.img_tf(Image.open(img_path).convert("RGB"))
        mask=self.mask_resize(Image.open(mask_path))
        mask=torch.as_tensor(np.array(mask),dtype=torch.long)
        mask=apply_global_mapping(mask,self.mapping)
        return img,mask

dataset=SUIMDataset(path)
train_size=int(0.8*len(dataset))
val_size=len(dataset)-train_size
train_ds,val_ds=random_split(dataset,[train_size,val_size])
train_loader=DataLoader(train_ds,batch_size=8,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=8)

def show_samples(ds,n=3):
    idxs=random.sample(range(len(ds)),n)
    for i in idxs:
        img,mask=ds[i]
        plt.figure(figsize=(8,3))
        plt.subplot(1,2,1)
        plt.imshow(img.permute(1,2,0))
        plt.axis("off")
        plt.subplot(1,2,2)
        plt.imshow(mask,vmin=0,vmax=7)
        plt.axis("off")
        plt.show()

show_samples(train_ds)


In [ ]:
# TO DO
#Had to pip the segmentation model. No idea why this is the case

!pip install segmentation_models_pytorch
import segmentation_models_pytorch as smp
import torch

model=smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8
)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)


In [ ]:
# TO DO
def train_one_epoch(model,loader,optimizer,criterion,device):
    model.train()
    total=0
    for imgs,masks in loader:
        imgs=imgs.to(device)
        masks=masks.to(device)
        optimizer.zero_grad()
        out=model(imgs)
        loss=criterion(out,masks)
        loss.backward()
        optimizer.step()
        total+=loss.item()*imgs.size(0)
    return total/len(loader.dataset)

def validate_one_epoch(model,loader,criterion,device):
    model.eval()
    with torch.no_grad():
      total=0
      for imgs,masks in loader:
          imgs=imgs.to(device)
          masks=masks.to(device)
          out=model(imgs)
          loss=criterion(out,masks)
          total+=loss.item()*imgs.size(0)
      return total/len(loader.dataset)


In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)

epochs = 10 #Ambitious with the 10 epochs but I'll be honest. If you run it with 3 it looks way too overfitted
train_losses = []
val_losses = []

for e in range(epochs):
    tl = train_one_epoch(model, train_loader, optimizer, criterion, device)
    vl = validate_one_epoch(model, val_loader, criterion, device)
    train_losses.append(tl)
    val_losses.append(vl)


plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()



In [ ]:
# TO DO
def visualize(model,loader):
    model.eval()
    with torch.no_grad():
      imgs,masks=next(iter(loader))
      imgs=imgs.to(device)
      preds=torch.argmax(model(imgs),dim=1).cpu()
      for i in range(3):
          plt.figure(figsize=(12,3))
          plt.subplot(1,3,1)
          plt.imshow(imgs[i].cpu().permute(1,2,0))
          plt.axis("off")
          plt.subplot(1,3,2)
          plt.imshow(masks[i])
          plt.axis("off")
          plt.subplot(1,3,3)
          plt.imshow(preds[i])
          plt.axis("off")
          plt.show()

visualize(model,val_loader)
